# UR10e Debugging helps 

First I need to see the Robot starting position and the rendering camera

In [1]:
import jax
import jax.numpy as jnp
import mediapy as media
from mujoco_playground._src import registry

# -------------------------
# Config
# -------------------------
env_name = "UR10PickCube"
seed = 0
episode_length = 250
render_every = 1
camera_name = "side_130"

# -------------------------
# Load env (eager reset for debugging)
# -------------------------
env = registry.load(env_name)

rng = jax.random.PRNGKey(seed)
state = env.reset(rng)

# -------------------------
# Print initial positions (works because reset is NOT jitted)
# -------------------------
robot_qpos = state.data.qpos[env._robot_qposadr]
box_pos = state.data.qpos[env._obj_qposadr : env._obj_qposadr + 3]
ctrl = state.data.ctrl
print(f"Robot qpos: {robot_qpos}")
print(f"Box pos:    {box_pos}")
print(f"Ctrl:       {ctrl}")

# -------------------------
# Action dimension (prefer env.action_size if available)
# -------------------------
if hasattr(env, "action_size"):
    act_dim = int(env.action_size)
elif hasattr(env, "action_dim"):
    act_dim = int(env.action_dim)
else:
    raise RuntimeError("Cannot infer action dimension")

zero_action = jnp.zeros((act_dim,), dtype=jnp.float32)

# -------------------------
# JIT step for rollout speed
# -------------------------
jit_step = jax.jit(env.step)

# -------------------------
# Rollout (jitted stepping)
# -------------------------
rollout = [state]
for _ in range(episode_length):
    state = jit_step(state, zero_action)
    rollout.append(state)

trajectory = rollout[::render_every]

# -------------------------
# Render inline (NO saving)
# -------------------------
frames = env.render(trajectory, camera=camera_name)
fps = float(1.0 / env.dt) / float(render_every)
media.show_video(frames, fps=fps)


Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
XML PATH: /Users/matthiasweiss/Desktop/ZHAW/MSE/3_VT1_Model Based RL/My_Mujoco/my_mujoco_playground/mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_single_cube_position.xml
✓ Using keyframe: 'low_home'
  Initial qpos size: 15
  Robot joints: [ 0.   -1.26  1.57 -1.95 -1.5  -1.5   0.    0.  ]
Model has keyframe support: True
[RESET] robot_qpos=[ 0.   -1.26  1.57 -1.95 -1.5  -1.5   0.    0.  ] | box_qpos=[ 0.60291755 -0.19164352  0.025     ] | ctrl=[ 0.   -1.26  1.57 -1.95 -1.5  -1.5   0.05] | ctrl-qpos-err=0.0
Robot qpos: [ 0.   -1.26  1.57 -1.95 -1.5  -1.5   0.    0.  ]
Box pos:    [ 0.60291755 -0.19164352  0.025     ]
Ctrl:       [ 0.   -1.26  1.57 -1.95 -1.5  -1.5   0.05]


/Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/jax/_src/interpreters/xla.py:119: RuntimeWarning: overflow encountered in cast
  return np.asarray(x, dtypes.canonicalize_dtype(x.dtype))
100%|██████████| 251/251 [00:02<00:00, 96.41it/s] 


This browser does not support the video tag.
